In [2]:
import mesa
import numpy as np


In [3]:
class VoterAgent(mesa.Agent):
    """An Voter Agent."""
    def __init__(self, id, model):
        super().__init__(id, model)
        # No Preferences Yet
        self.vote = None

    def step(self):
        # Decide who to vote for Randomly
        candidates = [c for c in self.model.schedule.agents if isinstance(c, CandidateAgent)]
        if candidates:
            self.vote = np.random.choice(candidates)
            print(f"Voter {self.unique_id} voted for Candidate {self.vote.unique_id}.")


In [4]:
class CandidateAgent(mesa.Agent):
    """A Candidate Agent."""
    def __init__(self, id, model):
        super().__init__(id, model)
        self.vote_count = 0

    def step(self):
        # No action yet for this version
        pass

In [ ]:
class ElectionModel(mesa.Model):
    """Election Model."""
    def __init__(self, num_voters, num_candidates):
        self.num_voters = num_voters
        self.num_candidates = num_candidates

        # Activate agents randomly
        self.schedule = mesa.time.RandomActivation(self)

        # Create Voters
        for i in range(self.num_voters):
            v = VoterAgent(f"v{i}", self)
            self.schedule.add(v)
        
        # Create Candidates
        for i in range(self.num_candidates):
            c = CandidateAgent(f"c{i}", self)
            self.schedule.add(c)

    def step(self):
        """Advance the model by one step."""

        # Reset vote counts
        for agent in self.schedule.agents:
            if isinstance(agent, CandidateAgent):
                agent.vote_count = 0

        # Run agent steps
        self.schedule.step()

        # Tally Votes
        self.tally_votes()

        # Determine Winner
        self.determine_winner()

    def tally_votes(self):
        """Tally votes for each candidate."""
        for agent in self.schedule.agents:
            if isinstance(agent, VoterAgent) and agent.vote is not None:
                agent.vote.vote_count += 1

    def determine_winner(self):
        """Determine the winner of the election (by Plurality)."""
        winner = None
        max_votes = -1
        candidates = [c for c in self.schedule.agents if isinstance(c, CandidateAgent)]
        for candidate in candidates:
            print(f"Candidate {candidate.unique_id} received {candidate.vote_count} votes.")
            if candidate.vote_count > max_votes:
                max_votes = candidate.vote_count
                winner = candidate

        if winner:
            print(f"The winner is Candidate {winner.unique_id} with {max_votes} votes.")
        else:
            print("No votes were cast.")
